# Portfolio recommendation (long-only, factor model)

One call to `stonks.optimize_portfolio`: fetch the last `MONTHS` months of the `TOP_N` most-liquid stocks, build a factor-model covariance (rank-`K` SVD + idiosyncratic diagonal), and solve the long-only mean-variance problem $\max\ \mu^\top w - \tfrac{\gamma}{2}w^\top\Sigma w$ with $w=\mathrm{softmax}(z)$. Returns the recommended holdings and weights.

> In-sample estimate — validate out-of-sample (see `portfolio_oos_test`). Not financial advice.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stonks import optimize_portfolio

%matplotlib inline


## Parameters


In [ ]:
TOP_N = 500
MONTHS = 3
GAMMA = 50.0      # risk aversion (higher = more diversified)
K = 25            # SVD factors; also the concentration dial
INTERVAL = "1d"


## Recommended portfolio


In [ ]:
weights = optimize_portfolio(top_n=TOP_N, months=MONTHS, interval=INTERVAL, gamma=GAMMA, K=K)
print(f"{len(weights)} holdings, sum(weight)={weights['weight'].sum():.4f}")
weights


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(weights)), weights["weight"])
ax.set_xticks(range(len(weights))); ax.set_xticklabels(weights.index, rotation=90, fontsize=7)
ax.set_ylabel("weight"); ax.set_title(rf"Long-only weights ($\gamma$={GAMMA}, K={K})")


## $\gamma$ sweep (concentration)

Higher $\gamma$ penalizes variance harder → more holdings, lower vol.


In [ ]:
print(f"{'gamma':>6} {'holdings':>9}")
for g in [2, 5, 10, 20, 50, 100]:
    w = optimize_portfolio(top_n=TOP_N, months=MONTHS, interval=INTERVAL, gamma=g, K=K)
    print(f"{g:6g} {len(w):9d}")


## Knobs

- **$\gamma$** — risk aversion. Higher → more diversified, lower vol.
- **$K$** — number of SVD factors, and the concentration dial: larger $K$ shrinks the idiosyncratic $\Psi$ → fewer, more concentrated holdings (too large reabsorbs Marchenko-Pastur noise).
- **months / top_n** — window length and universe size.

Same thing from the command line:

```
stonks portfolio --top-n 500 --months 3 --gamma 50 -k 25
```
